In [2]:
import pandas as pd
import numpy as np
import sys
# from scipy.fft import rfft, rfftfreq
#
# from sklearn.model_selection import StratifiedGroupKFold
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.svm import SVC
# from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix
# from sklearn.decomposition import PCA
#
# from sklearn.metrics import (
#     accuracy_score,
#     balanced_accuracy_score,
#     precision_score,
#     recall_score,
#     f1_score,
# )

In [3]:
df = pd.read_csv('../data/df_processed.csv')

In [9]:
RECORD_COLS = ["set_id", "subject", "exercise", "set_nr", "focus"]
def make_windows(df, window=200, stride=75, decim=2):
    sensor_cols = [c for c in df.columns if c not in RECORD_COLS + ["time"]]
    records = df[RECORD_COLS].drop_duplicates().reset_index(drop=True)
    # robust 0..k-1 exercise codes (handles string / 1-indexed labels)
    ex_codes = pd.factorize(records["exercise"])[0]

    Xw, win_rec, win_exercise = [], [], []
    for ridx, row in records.iterrows():
        g = df[df["set_id"] == row["set_id"]].sort_values("time")
        arr = g[sensor_cols].to_numpy(dtype=np.float32)
        # samples every decim value, cutting the amount of records.
        if decim > 1:
            arr = arr[::decim]
        # length of records after downsampling
        L = arr.shape[0]
        for s in range(0, max(1, L - window + 1), stride):
            w = arr[s:s + window]
            # in case last window is shorter, don't add it
            if w.shape[0] < window:
                continue
            Xw.append(w)
            win_rec.append(ridx)
            win_exercise.append(int(ex_codes[ridx]))

    return (records, sensor_cols,
            np.stack(Xw), # window count * sets, window length, features
            np.asarray(win_rec, dtype=np.int64),
            np.asarray(win_exercise, dtype=np.int64))


records, sensor_cols, Xw, win_rec, win_exercise = make_windows(df)

In [30]:
win_rec

array([ 0,  0,  0,  0,  0,  0,  0,  0,  1,  1,  1,  1,  1,  1,  1,  1,  2,
        2,  2,  2,  2,  2,  2,  2,  3,  3,  3,  3,  3,  3,  3,  3,  4,  4,
        4,  4,  4,  4,  4,  4,  5,  5,  5,  5,  5,  5,  5,  5,  6,  6,  6,
        6,  6,  6,  6,  6,  7,  7,  7,  7,  7,  7,  7,  7,  8,  8,  8,  8,
        8,  8,  8,  8,  9,  9,  9,  9,  9,  9,  9,  9, 10, 10, 10, 10, 10,
       10, 10, 10, 11, 11, 11, 11, 11, 11, 11, 11, 12, 12, 12, 12, 12, 12,
       12, 12, 13, 13, 13, 13, 13, 13, 13, 13, 14, 14, 14, 14, 14, 14, 14,
       14, 15, 15, 15, 15, 15, 15, 15, 15, 16, 16, 16, 16, 16, 16, 16, 16,
       17, 17, 17, 17, 17, 17, 17, 17, 18, 18, 18, 18, 18, 18, 18, 18, 19,
       19, 19, 19, 19, 19, 19, 19, 20, 20, 20, 20, 20, 20, 20, 20, 21, 21,
       21, 21, 21, 21, 21, 21, 22, 22, 22, 22, 22, 22, 22, 22, 23, 23, 23,
       23, 23, 23, 23, 23])

In [27]:
sensor_cols = [c for c in df.columns if c not in RECORD_COLS + ["time"]]
len(sensor_cols)
192/8

24.0

In [4]:
from scipy.fft import rfft, rfftfreq
import numpy as np

SAMPLE_RATE = 50  # 20 ms intervals

sensor_cols = [c for c in df.columns if c not in ["time", "subject", "exercise", "set_nr", "focus", "set_id"]]

def fft_features(x):
    x = np.asarray(x)

    fft_vals = np.abs(rfft(x))
    freqs = rfftfreq(len(x), d=1 / SAMPLE_RATE)

    fft_vals[0] = 0

    dominant_idx = np.argmax(fft_vals)

    dominant_freq = freqs[dominant_idx]
    dominant_power = fft_vals[dominant_idx]

    p = fft_vals / (fft_vals.sum() + 1e-12)

    spectral_entropy = -(p * np.log2(p + 1e-12)).sum()
    spectral_centroid = np.sum(freqs * fft_vals) / (np.sum(fft_vals) + 1e-12)

    return {
        "dominant_freq": dominant_freq,
        "dominant_power": dominant_power,
        "spectral_entropy": spectral_entropy,
        "spectral_centroid": spectral_centroid,
    }


fft_rows = []

for set_id, group in df.groupby("set_id"):
    row = {
        "set_id": set_id,
        "subject": group["subject"].iloc[0],
        "exercise": group["exercise"].iloc[0],
        "set_nr": group["set_nr"].iloc[0],
        "focus": group["focus"].iloc[0],
    }

    for col in sensor_cols:
        feats = fft_features(group[col])

        for feat_name, value in feats.items():
            row[f"{col}_{feat_name}"] = value

    fft_rows.append(row)

fft_df = pd.DataFrame(fft_rows)

In [97]:


def add_axis_combinations(g):
    AXIS_TRIPLES = {
        "acc_lin": ["acc_lin_x_lowpass", "acc_lin_y_lowpass", "acc_lin_z_lowpass"],
        "acc":     ["acc_x_lowpass", "acc_y_lowpass", "acc_z_lowpass"],
        "gyro":    ["gyro_x_lowpass", "gyro_y_lowpass", "gyro_z_lowpass"],
        "orient":  ["yaw_lowpass", "pitch_lowpass", "roll_lowpass"],
    }
    new_data = {}

    for name, (cx, cy, cz) in AXIS_TRIPLES.items():
        x, y, z = g[cx], g[cy], g[cz]

        new_data[f"{name}_magnitude"] = np.sqrt(x**2 + y**2 + z**2)
        # new_data[f"{name}_sum"] = x + y + z
        new_data[f"{name}_abs_sum"] = x.abs() + y.abs() + z.abs()
        eps = 1e-8
        new_data[f"{name}_pairwise_product_sum"] = x*y + x*z + y*z


    new_df = pd.DataFrame(new_data, index=g.index)
    return pd.concat([g, new_df], axis=1), list(new_data.keys())


In [98]:
df, newcols = add_axis_combinations(df)

sensor_cols = [c for c in df.columns if c not in ["time", "subject", "exercise", "set_nr", "focus", "set_id"]]

agg_df = (
    df.groupby("set_id")[sensor_cols].agg(["std", "min", "max", "median"])
)

agg_df.columns = [f"{col}_{stat}" for col, stat in agg_df.columns]

agg_df = agg_df.reset_index()

svm_df = agg_df.merge(fft_df, on="set_id", how="inner")

In [58]:
print(
    len([c for c in X.columns if c.startswith("acc_lin_")]),
    len([c for c in X.columns if c.startswith(("yaw_", "pitch_", "roll_", "orient_"))]),
    len([c for c in X.columns if c.startswith("gyro_")]),
    len([c for c in X.columns if c.startswith("acc_") and not c.startswith("acc_lin_")]),
    len([c for c in X.columns if c.startswith("hr_")]),
)

36 36 36 36 8


In [101]:

# testing how much PCA components to use
groups = {
    "acc_lin": [c for c in svm_df.columns if c.startswith("acc_lin_")],
    "acc":     [c for c in svm_df.columns if c.startswith("acc_") and not c.startswith("acc_lin_")],
    "gyro":    [c for c in svm_df.columns if c.startswith("gyro_")],
    "orient":  [c for c in svm_df.columns if c.startswith(("yaw_", "pitch_", "roll_", "orient_"))],
    "hr":      [c for c in svm_df.columns if c.startswith("hr_")]
}

pca_df = pd.DataFrame(index=svm_df.index)

for group_name, cols in groups.items():
    X_group = StandardScaler().fit_transform(svm_df[cols])

    n_comp = 3 if group_name == "hr" else 6

    pca = PCA(n_components=n_comp)
    pcs = pca.fit_transform(X_group)

    for i in range(n_comp):
        pca_df[f"{group_name}_pc{i+1}"] = pcs[:, i]

    print(
        group_name,
        "explained variance:",
        round(pca.explained_variance_ratio_.sum(), 3)
    )

acc_lin explained variance: 0.841
acc explained variance: 0.886
gyro explained variance: 0.852
orient explained variance: 0.907
hr explained variance: 0.982


In [74]:
from itertools import product

In [1]:
def get_splits(data):
    split_groups = []

    for subject in data["subject"].unique():
        for focus in data["focus"].unique():
            candidates = data[
                (data["subject"] == subject) &
                (data["focus"] == focus)
            ].index.to_numpy()

            split_groups.append(candidates)

    all_test_splits = [np.array(split) for split in product(*split_groups)]
    base_rng.shuffle(all_test_splits)

    print("Total possible unique splits:", len(all_test_splits))
    print("Using splits:", n_repeats)

    return all_test_splits[:min(9999, len(all_test_splits))]


In [102]:
results = []
all_predictions = []

data = svm_df.copy()
n_repeats = 100
base_rng = np.random.default_rng(42)
meta_cols = ["set_id", "subject", "exercise", "set_nr", "focus"]
target_col = "focus"


selected_test_splits = get_splits(data)

for repeat in range(n_repeats):
    if repeat % 50 == 0:
        print(f"progress: {repeat/n_repeats*100}%")

    test_idx = selected_test_splits[repeat]
    train_idx = data.index.difference(test_idx).to_numpy()

    train_df_raw = data.loc[train_idx].copy()
    test_df_raw = data.loc[test_idx].copy()

    feature_cols = [c for c in data.columns if c not in meta_cols]

    X_train_raw = train_df_raw[feature_cols].copy()
    X_test_raw = test_df_raw[feature_cols].copy()

    y_train = train_df_raw[target_col]
    y_test = test_df_raw[target_col]

    groups = {
        "acc_lin": [c for c in feature_cols if c.startswith("acc_lin_")],
        "acc": [c for c in feature_cols if c.startswith("acc_") and not c.startswith("acc_lin_")],
        "gyro": [c for c in feature_cols if c.startswith("gyro_")],
        "orient": [c for c in feature_cols if c.startswith(("yaw_", "pitch_", "roll_", "orient_"))],
        "hr": [c for c in feature_cols if c.startswith("hr_")],
    }

    train_pca_df = pd.DataFrame(index=train_df_raw.index)
    test_pca_df = pd.DataFrame(index=test_df_raw.index)

    for group_name, cols in groups.items():
        if len(cols) == 0:
            continue

        n_comp = 3 if group_name == "hr" else 6
        n_comp = min(n_comp, len(cols), len(train_df_raw) - 1)

        scaler = StandardScaler()
        pca = PCA(n_components=n_comp)

        X_train_group_scaled = scaler.fit_transform(X_train_raw[cols])
        X_test_group_scaled = scaler.transform(X_test_raw[cols])

        train_pcs = pca.fit_transform(X_train_group_scaled)
        test_pcs = pca.transform(X_test_group_scaled)

        for i in range(n_comp):
            train_pca_df[f"{group_name}_pc{i+1}"] = train_pcs[:, i]
            test_pca_df[f"{group_name}_pc{i+1}"] = test_pcs[:, i]

    train_extra = train_df_raw[["exercise", "set_nr"]].copy()
    test_extra = test_df_raw[["exercise", "set_nr"]].copy()

    train_extra = pd.get_dummies(train_extra, columns=["exercise"], prefix="exercise", dtype=int)
    test_extra = pd.get_dummies(test_extra, columns=["exercise"], prefix="exercise", dtype=int)

    test_extra = test_extra.reindex(columns=train_extra.columns, fill_value=0)

    X_train = pd.concat([train_pca_df, train_extra], axis=1)
    X_test = pd.concat([test_pca_df, test_extra], axis=1)

    svm = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="rbf", C=1, gamma="scale", class_weight="balanced"))
    ])

    svm.fit(X_train, y_train)
    y_pred = svm.predict(X_test)

    results.append({
        "repeat": repeat,
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "n_train": len(train_idx),
        "n_test": len(test_idx),
    })

    pred_df = test_df_raw[["set_id", "subject", "exercise", "set_nr", "focus"]].copy()
    pred_df["repeat"] = repeat
    pred_df["pred"] = y_pred
    pred_df["correct"] = pred_df["focus"] == pred_df["pred"]
    all_predictions.append(pred_df)


results_df = pd.DataFrame(results)
predictions_df = pd.concat(all_predictions, ignore_index=True)

metric_cols = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1"
]

print(results_df[metric_cols].describe())

print("\nMean accuracy:", results_df["accuracy"].mean())
print("Mean balanced accuracy:", results_df["balanced_accuracy"].mean())

print("\nPrediction counts by record:")
print(
    predictions_df
    .groupby(["set_id", "subject", "exercise", "set_nr", "focus"])
    ["correct"]
    .agg(["count", "mean"])
    .sort_values("mean")
)

Total possible unique splits: 1296
Using splits: 1296
progress: 0.0%
progress: 3.8580246913580245%
progress: 7.716049382716049%
progress: 11.574074074074074%
progress: 15.432098765432098%
progress: 19.290123456790123%
progress: 23.14814814814815%
progress: 27.00617283950617%
progress: 30.864197530864196%
progress: 34.72222222222222%
progress: 38.58024691358025%
progress: 42.43827160493827%
progress: 46.2962962962963%
progress: 50.15432098765432%
progress: 54.01234567901234%
progress: 57.870370370370374%
progress: 61.72839506172839%
progress: 65.58641975308642%
progress: 69.44444444444444%
progress: 73.30246913580247%
progress: 77.1604938271605%
progress: 81.01851851851852%
progress: 84.87654320987654%
progress: 88.73456790123457%
progress: 92.5925925925926%
progress: 96.4506172839506%
          accuracy  balanced_accuracy    precision       recall           f1
count  1296.000000        1296.000000  1296.000000  1296.000000  1296.000000
mean      0.624807           0.624807     0.626157